In [5]:
# =============================================================================
# BLOCO 1 — SPLIT DE DADOS SEGURO E MOTOR DE INGESTÃO (DATASET & DATALOADER)
# =============================================================================
import os
import torch
import torchaudio
import numpy as np
import pandas as pd
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de processamento: {DEVICE}")

# --- 1. O SPLIT DE DADOS CONTRASTIVO (80% / 10% / 10%) ---
print("\nRealizando Particionamento de Dados Seguro (Sem Vazamento de Pares)...")
CSV_PATH = "dataset_fase2_valid_elite/metadata_elite_treino.csv"
df_total = pd.read_csv(CSV_PATH)

# Sorteamos e dividimos os PARES (e não as linhas avulsas)
pares_unicos = df_total['hard_negative_pair_id'].unique()
np.random.seed(42) # Semente fixa para reprodutibilidade no TCC
np.random.shuffle(pares_unicos)

n_pares = len(pares_unicos)
train_ids = pares_unicos[:int(n_pares * 0.8)]
val_ids = pares_unicos[int(n_pares * 0.8):int(n_pares * 0.9)]
test_ids = pares_unicos[int(n_pares * 0.9):]

df_train = df_total[df_total['hard_negative_pair_id'].isin(train_ids)].reset_index(drop=True)
df_val = df_total[df_total['hard_negative_pair_id'].isin(val_ids)].reset_index(drop=True)
df_test = df_total[df_total['hard_negative_pair_id'].isin(test_ids)].reset_index(drop=True)

print(f"  -> Treino:    {len(df_train)} amostras")
print(f"  -> Validação: {len(df_val)} amostras")
print(f"  -> Teste:     {len(df_test)} amostras")

# --- 2. O DATASET MODIFICADO ---
class TccTimbreDataset(Dataset):
    # Agora aceita o DataFrame já filtrado em vez do caminho do CSV
    def __init__(self, df_particionado, audio_dir, processor_name="laion/clap-htsat-fused"):
        self.df = df_particionado
        self.audio_dir = audio_dir
        self.processor = AutoProcessor.from_pretrained(processor_name)
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['file_name'])
        texto = row['text']
        
        audio_np, sr = sf.read(audio_path)
        if audio_np.ndim == 1: waveform = torch.from_numpy(audio_np).unsqueeze(0).float()
        else: waveform = torch.from_numpy(audio_np).t().float()
        
        if sr != 48000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=48000)
            waveform = resampler(waveform)
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        return {
            "waveform": waveform.squeeze().numpy(), 
            "text": texto,
            "keyword": row['keyword'],
            "hard_negative_id": row['hard_negative_pair_id']
        }

def collate_fn(batch, processor):
    waveforms = [item["waveform"] for item in batch]
    texts = [item["text"] for item in batch]
    inputs = processor(audio=waveforms, text=texts, return_tensors="pt", padding=True, sampling_rate=48000)
    inputs["hard_negative_id"] = [item["hard_negative_id"] for item in batch]
    return inputs

Dispositivo de processamento: cpu

Realizando Particionamento de Dados Seguro (Sem Vazamento de Pares)...
  -> Treino:    4580 amostras
  -> Validação: 570 amostras
  -> Teste:     580 amostras


C:\Users\victo\AppData\Local\Temp\ipykernel_3852\547657913.py:24: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(pares_unicos)


In [6]:
# =============================================================================
# BLOCO 1 — MOTOR DE INGESTÃO MULTIMODAL (DATASET & DATALOADER)
# =============================================================================
import os
import torch
import torchaudio
import pandas as pd
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor

# Configuração de dispositivo (Usa GPU se disponível)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de processamento: {DEVICE}")

class TccTimbreDataset(Dataset):
    def __init__(self, csv_path, audio_dir, processor_name="laion/clap-htsat-fused"):
        print(f"Carregando metadados de: {csv_path}")
        self.df = pd.read_csv(csv_path)
        self.audio_dir = audio_dir
        
        # O Processor do CLAP é um "2 em 1": 
        # Ele tokeniza o texto (RoBERTa) e extrai o Mel-Espectrograma do áudio (HTSAT)
        print("Carregando o AutoProcessor do CLAP...")
        self.processor = AutoProcessor.from_pretrained(processor_name)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['file_name'])
        texto = row['text']
        
        # 1. Carregamento Nativo e Seguro via Soundfile (Evita erro do torchcodec)
        audio_np, sr = sf.read(audio_path)
        
        # Converte o numpy array para Tensor do PyTorch
        # O PyTorch exige (canais, frames)
        if audio_np.ndim == 1:
            waveform = torch.from_numpy(audio_np).unsqueeze(0).float()
        else:
            waveform = torch.from_numpy(audio_np).t().float()
        
        # 2. Conversão Obrigatória para 48kHz (Exigência do LAION-CLAP)
        if sr != 48000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=48000)
            waveform = resampler(waveform)
            
        # 3. Força para canal Mono (caso tenha vindo estéreo)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        return {
            "waveform": waveform.squeeze().numpy(), # O Processor do HF espera um array 1D
            "text": texto,
            "keyword": row['keyword'],
            "hard_negative_id": row['hard_negative_pair_id']
        }

def collate_fn(batch, processor):
    """
    Esta função pega os itens isolados do Dataset e empacota em um "Lote" (Batch)
    para mandar pra placa de vídeo de forma otimizada.
    """
    waveforms = [item["waveform"] for item in batch]
    texts = [item["text"] for item in batch]
    
    # O processor faz o padding dos áudios e textos para que todos tenham o mesmo tamanho no lote
    inputs = processor(
        audio=waveforms, # <--- MUDEI DE "audios=" PARA "audio=" AQUI
        text=texts, 
        return_tensors="pt", 
        padding=True, 
        sampling_rate=48000
    )
    
    # Preservamos as chaves para a lógica do Hard Negative na função de Loss depois
    inputs["hard_negative_id"] = [item["hard_negative_id"] for item in batch]
    
    return inputs

# =============================================================================
# TESTE DE SANIDADE DO DATALOADER
# =============================================================================
if __name__ == "__main__":
    # Aponte para a sua pasta de elite gerada
    CSV_PATH = "dataset_fase2_valid_elite/metadata_elite_treino.csv"
    AUDIO_DIR = "dataset_fase2_valid_elite/audio"
    
    # Cria o dataset
    dataset = TccTimbreDataset(CSV_PATH, AUDIO_DIR)
    
    # Cria o Dataloader (Batch size de 4 apenas para o teste inicial não estourar a RAM)
    dataloader = DataLoader(
        dataset, 
        batch_size=4, 
        shuffle=True, 
        collate_fn=lambda b: collate_fn(b, dataset.processor)
    )
    
    # Puxa apenas o primeiro lote para ver se a matemática está certa
    print("\nProcessando o primeiro lote (Isso pode levar alguns segundos na primeira vez)...")
    lote_teste = next(iter(dataloader))
    
    print("\n✓ Teste de Sanidade do Lote:")
    print(f"Shape dos tensores de Áudio: {lote_teste['input_features'].shape}")
    print(f"Shape dos tensores de Texto: {lote_teste['input_ids'].shape}")
    print(f"IDs de Hard Negatives no lote: {lote_teste['hard_negative_id']}")

Dispositivo de processamento: cpu
Carregando metadados de: dataset_fase2_valid_elite/metadata_elite_treino.csv
Carregando o AutoProcessor do CLAP...

Processando o primeiro lote (Isso pode levar alguns segundos na primeira vez)...

✓ Teste de Sanidade do Lote:
Shape dos tensores de Áudio: torch.Size([4, 4, 1001, 64])
Shape dos tensores de Texto: torch.Size([4, 13])
IDs de Hard Negatives no lote: ['par_00098', 'par_00947', 'par_00532', 'par_00195']


In [7]:
# =============================================================================
# BLOCO 2 — ARQUITETURA DO MODELO (TREINAMENTO DO ZERO - SEM DATA LEAKAGE)
# =============================================================================
from transformers import ClapModel
from peft import LoraConfig, get_peft_model

print("1. Baixando o Modelo Fundacional (LAION-CLAP)...")
modelo_base = ClapModel.from_pretrained("laion/clap-htsat-fused")
modelo_base = modelo_base.to(DEVICE)

print("\n2. Configurando as Matrizes LoRA (Iniciando do Zero)...")
configuracao_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["logit_scale_a", "logit_scale_t"]
)

modelo_especializado = get_peft_model(modelo_base, configuracao_lora)

print("\n✓ Arquitetura Pronta e Virgem! Zero risco de Data Leakage.")
print("=" * 60)
modelo_especializado.print_trainable_parameters()
print("=" * 60)

1. Baixando o Modelo Fundacional (LAION-CLAP)...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 477/477 [00:00<00:00, 3373.21it/s]



2. Configurando as Matrizes LoRA (Iniciando do Zero)...

✓ Arquitetura Pronta e Virgem! Zero risco de Data Leakage.
trainable params: 436,224 || all params: 153,943,754 || trainable%: 0.2834


In [8]:
# =============================================================================
# BLOCO 3 — TREINO, VALIDAÇÃO E TESTE (SMART BATCH + EARLY STOPPING VERDADEIRO)
# =============================================================================
import random
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict
from torch.optim import AdamW
from torch.utils.data import Sampler, DataLoader
from tqdm import tqdm

# --- O JUIZ (AGORA AVALIANDO A VALIDAÇÃO) ---
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.015, save_dir="tcc_clap_lora_timbre_elite"):
        self.patience = patience
        self.min_delta = min_delta
        self.save_dir = save_dir
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model, processor):
        if val_loss < self.best_loss - self.min_delta:
            print(f"  🌟 Novo Ponto de Convergência! Val Loss caiu para {val_loss:.4f}.")
            print(f"  💾 Salvando Pesos Ouro no HD...")
            self.best_loss = val_loss
            self.counter = 0
            model.save_pretrained(self.save_dir)
            processor.save_pretrained(self.save_dir)
        else:
            self.counter += 1
            print(f"  ⚠️ Val Loss estagnou ou subiu. Paciência: {self.counter}/{self.patience}")
            if self.counter >= self.patience: self.early_stop = True

# --- O MAESTRO DOS LOTES ---
class HardNegativeBatchSampler(Sampler):
    def __init__(self, dataset, batch_size):
        self.dataset = dataset
        self.batch_size = batch_size
        self.pair_dict = defaultdict(list)
        for idx, row in dataset.df.iterrows():
            self.pair_dict[row['hard_negative_pair_id']].append(idx)
        self.pair_ids = list(self.pair_dict.keys())
        
    def __iter__(self):
        random.shuffle(self.pair_ids)
        batch = []
        for pair_id in self.pair_ids:
            batch.extend(self.pair_dict[pair_id])
            if len(batch) >= self.batch_size:
                yield batch[:self.batch_size]
                batch = batch[self.batch_size:]
        if len(batch) > 0: yield batch

    def __len__(self): return (len(self.dataset) + self.batch_size - 1) // self.batch_size

# --- A FUNÇÃO DE AVALIAÇÃO CIENTÍFICA (SEM GRADIENTES) ---
def avaliar_modelo(modelo, dataloader_eval):
    modelo.eval() # Congela Dropout e BatchNorm
    loss_acumulada = 0.0
    
    with torch.no_grad(): # Desliga o motor de gradientes para não vazar aprendizado
        for batch in dataloader_eval:
            input_features = batch["input_features"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            
            batch_size_atual = input_features.shape[0]
            is_longer = torch.zeros((batch_size_atual, 1), dtype=torch.bool, device=DEVICE) 
            
            outputs = modelo(
                input_features=input_features, input_ids=input_ids,
                attention_mask=attention_mask, is_longer=is_longer
            )
            
            labels = torch.arange(batch_size_atual, device=DEVICE)
            loss_a = F.cross_entropy(outputs.logits_per_audio, labels)
            loss_t = F.cross_entropy(outputs.logits_per_text, labels)
            loss = (loss_a + loss_t) / 2.0
            loss_acumulada += loss.item()
            
    return loss_acumulada / len(dataloader_eval)

# --- CONFIGURAÇÕES ---
BATCH_SIZE = 32       
EPOCHS = 15            
LEARNING_RATE = 1e-4   # Voltamos para a LR original de inicio de treino
AUDIO_DIR = "dataset_fase2_valid_elite/audio"

print("\n1. Instanciando 3 Dataloaders (Treino, Validação, Teste)...")
ds_train = TccTimbreDataset(df_train, AUDIO_DIR)
ds_val = TccTimbreDataset(df_val, AUDIO_DIR)
ds_test = TccTimbreDataset(df_test, AUDIO_DIR)

dl_train = DataLoader(ds_train, batch_sampler=HardNegativeBatchSampler(ds_train, BATCH_SIZE), collate_fn=lambda b: collate_fn(b, ds_train.processor))
dl_val = DataLoader(ds_val, batch_sampler=HardNegativeBatchSampler(ds_val, BATCH_SIZE), collate_fn=lambda b: collate_fn(b, ds_val.processor))
dl_test = DataLoader(ds_test, batch_sampler=HardNegativeBatchSampler(ds_test, BATCH_SIZE), collate_fn=lambda b: collate_fn(b, ds_test.processor))

print("2. Configurando Otimizador e Monitor de Validação...")
otimizador = AdamW(modelo_especializado.parameters(), lr=LEARNING_RATE)
monitor_parada = EarlyStopping(patience=3, min_delta=0.015, save_dir="tcc_clap_lora_timbre_elite")

print("\n" + "="*80)
print("🚀 INICIANDO TREINAMENTO COM VALIDAÇÃO (EPOCH 6 EM DIANTE)")
print("="*80)

for epoch in range(EPOCHS):
    modelo_especializado.train()
    loss_train_acumulada = 0
    
    loop = tqdm(dl_train, leave=False, desc=f"Epoch {epoch+1}/{EPOCHS} [Treino]")
    
    for batch in loop:
        otimizador.zero_grad()
        
        input_features = batch["input_features"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        batch_size_atual = input_features.shape[0]
        is_longer = torch.zeros((batch_size_atual, 1), dtype=torch.bool, device=DEVICE) 
        
        outputs = modelo_especializado(input_features=input_features, input_ids=input_ids, attention_mask=attention_mask, is_longer=is_longer)
        
        labels = torch.arange(batch_size_atual, device=DEVICE)
        loss = (F.cross_entropy(outputs.logits_per_audio, labels) + F.cross_entropy(outputs.logits_per_text, labels)) / 2.0
        
        loss.backward()
        otimizador.step()
        loss_train_acumulada += loss.item()
        loop.set_postfix(train_loss=f"{loss.item():.4f}")
        
    loss_train = loss_train_acumulada / len(dl_train)
    
    # ------------------ FASE DE VALIDAÇÃO ------------------
    print(f"\n-> Fim da Epoch {epoch+1} | Calculando Validation Loss...")
    loss_val = avaliar_modelo(modelo_especializado, dl_val)
    
    print(f"   Train Loss: {loss_train:.4f}  |  Val Loss: {loss_val:.4f}")
    
    # O Early Stopping agora é acionado PELA VALIDAÇÃO!
    monitor_parada(loss_val, modelo_especializado, ds_train.processor)
    
    if monitor_parada.early_stop:
        print("\n" + "=" * 80)
        print("🛑 EARLY STOPPING ACIONADO: O modelo convergiu no conjunto de Validação.")
        print("=" * 80)
        break

# =============================================================================
# FASE FINAL: O TESTE CEGO ABSOLUTO
# =============================================================================
print("\n" + "="*80)
print("🎯 FASE DE TESTE CEGO: AVALIANDO O MELHOR MODELO SALVO")
print("="*80)

from peft import PeftModel
print("Recarregando os Pesos Ouro da pasta definitiva...")
modelo_definitivo = PeftModel.from_pretrained(modelo_base, monitor_parada.save_dir, is_trainable=False)

loss_test = avaliar_modelo(modelo_definitivo, dl_test)

print(f"\n🏆 RESULTADO FINAL PARA O TCC:")
print(f"   Loss em Dados de Teste (Nunca Vistos): {loss_test:.4f}")
print("=" * 80)


1. Instanciando 3 Dataloaders (Treino, Validação, Teste)...
Carregando metadados de:                                               file_name  \
0     eq_bright_20pct_keyboard_electronic_069-070-02...   
1     eq_bright_40pct_keyboard_electronic_069-070-02...   
2     eq_bright_60pct_keyboard_electronic_069-070-02...   
3     eq_bright_80pct_keyboard_electronic_069-070-02...   
4     eq_bright_100pct_keyboard_electronic_069-070-0...   
...                                                 ...   
4575    eq_muffled_20pct_vocal_acoustic_000-054-127.wav   
4576    eq_muffled_40pct_vocal_acoustic_000-054-127.wav   
4577    eq_muffled_60pct_vocal_acoustic_000-054-127.wav   
4578    eq_muffled_80pct_vocal_acoustic_000-054-127.wav   
4579   eq_muffled_100pct_vocal_acoustic_000-054-127.wav   

                                                   text  keyword effect_type  \
0           keyboard with a faintly bright equalization   bright          eq   
1     a recording of a keyboard modified to s

TypeError: argument of type 'method' is not a container or iterable

In [1]:
import torch

print(f"GPU disponível? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nome da Placa: {torch.cuda.get_device_name(0)}")

GPU disponível? True
Nome da Placa: NVIDIA GeForce RTX 4050 Laptop GPU


In [9]:
# =============================================================================
# PIPELINE DE TREINAMENTO DEFINITIVO (SPLIT SEGURO + LORA ZERO + EARLY STOPPING)
# =============================================================================
import os
import random
import torch
import torchaudio
import numpy as np
import pandas as pd
import soundfile as sf
import torch.nn.functional as F
from collections import defaultdict
from torch.optim import AdamW
from torch.utils.data import Dataset, Sampler, DataLoader
from transformers import AutoProcessor, ClapModel
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de processamento: {DEVICE}")
print("=" * 80)

# =============================================================================
# 1. CLASSES E FUNÇÕES CORE
# =============================================================================

class TccTimbreDataset(Dataset):
    def __init__(self, df_particionado, audio_dir, processor_name="laion/clap-htsat-fused"):
        self.df = df_particionado
        self.audio_dir = audio_dir
        self.processor = AutoProcessor.from_pretrained(processor_name)
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['file_name'])
        texto = row['text']
        
        audio_np, sr = sf.read(audio_path)
        if audio_np.ndim == 1: waveform = torch.from_numpy(audio_np).unsqueeze(0).float()
        else: waveform = torch.from_numpy(audio_np).t().float()
        
        if sr != 48000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=48000)
            waveform = resampler(waveform)
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        return {
            "waveform": waveform.squeeze().numpy(), 
            "text": texto,
            "keyword": row['keyword'],
            "hard_negative_id": row['hard_negative_pair_id']
        }

def collate_fn(batch, processor):
    waveforms = [item["waveform"] for item in batch]
    texts = [item["text"] for item in batch]
    inputs = processor(audio=waveforms, text=texts, return_tensors="pt", padding=True, sampling_rate=48000)
    inputs["hard_negative_id"] = [item["hard_negative_id"] for item in batch]
    return inputs

class HardNegativeBatchSampler(Sampler):
    def __init__(self, dataset, batch_size):
        self.dataset = dataset
        self.batch_size = batch_size
        self.pair_dict = defaultdict(list)
        for idx, row in dataset.df.iterrows():
            self.pair_dict[row['hard_negative_pair_id']].append(idx)
        self.pair_ids = list(self.pair_dict.keys())
        
    def __iter__(self):
        random.shuffle(self.pair_ids)
        batch = []
        for pair_id in self.pair_ids:
            batch.extend(self.pair_dict[pair_id])
            if len(batch) >= self.batch_size:
                yield batch[:self.batch_size]
                batch = batch[self.batch_size:]
        if len(batch) > 0: yield batch

    def __len__(self): return (len(self.dataset) + self.batch_size - 1) // self.batch_size

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.015, save_dir="tcc_clap_lora_timbre_elite"):
        self.patience = patience
        self.min_delta = min_delta
        self.save_dir = save_dir
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model, processor):
        if val_loss < self.best_loss - self.min_delta:
            print(f"  🌟 Novo Ponto de Convergência! Val Loss caiu para {val_loss:.4f}.")
            print(f"  💾 Salvando Pesos Ouro no HD...")
            self.best_loss = val_loss
            self.counter = 0
            model.save_pretrained(self.save_dir)
            processor.save_pretrained(self.save_dir)
        else:
            self.counter += 1
            print(f"  ⚠️ Val Loss estagnou ou subiu. Paciência: {self.counter}/{self.patience}")
            if self.counter >= self.patience: self.early_stop = True

def avaliar_modelo(modelo, dataloader_eval):
    modelo.eval()
    loss_acumulada = 0.0
    with torch.no_grad():
        for batch in dataloader_eval:
            input_features = batch["input_features"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            batch_size_atual = input_features.shape[0]
            is_longer = torch.zeros((batch_size_atual, 1), dtype=torch.bool, device=DEVICE) 
            
            outputs = modelo(input_features=input_features, input_ids=input_ids, attention_mask=attention_mask, is_longer=is_longer)
            labels = torch.arange(batch_size_atual, device=DEVICE)
            loss_a = F.cross_entropy(outputs.logits_per_audio, labels)
            loss_t = F.cross_entropy(outputs.logits_per_text, labels)
            loss = (loss_a + loss_t) / 2.0
            loss_acumulada += loss.item()
    return loss_acumulada / len(dataloader_eval)

# =============================================================================
# 2. DATA SPLIT SEGURO (SEM DATA LEAKAGE)
# =============================================================================
print("\nRealizando Particionamento de Dados Seguro (Sem Vazamento de Pares)...")
CSV_PATH = "dataset_fase2_valid_elite/metadata_elite_treino.csv"
AUDIO_DIR = "dataset_fase2_valid_elite/audio"

df_total = pd.read_csv(CSV_PATH)
pares_unicos = df_total['hard_negative_pair_id'].unique()
np.random.seed(42)
np.random.shuffle(pares_unicos)

n_pares = len(pares_unicos)
train_ids = pares_unicos[:int(n_pares * 0.8)]
val_ids = pares_unicos[int(n_pares * 0.8):int(n_pares * 0.9)]
test_ids = pares_unicos[int(n_pares * 0.9):]

df_train = df_total[df_total['hard_negative_pair_id'].isin(train_ids)].reset_index(drop=True)
df_val = df_total[df_total['hard_negative_pair_id'].isin(val_ids)].reset_index(drop=True)
df_test = df_total[df_total['hard_negative_pair_id'].isin(test_ids)].reset_index(drop=True)

print(f"  -> Treino:    {len(df_train)} amostras")
print(f"  -> Validação: {len(df_val)} amostras")
print(f"  -> Teste:     {len(df_test)} amostras")

BATCH_SIZE = 32       
EPOCHS = 15            
LEARNING_RATE = 1e-4   

print("\nInstanciando Dataloaders (Treino, Validação, Teste)...")
ds_train = TccTimbreDataset(df_train, AUDIO_DIR)
ds_val = TccTimbreDataset(df_val, AUDIO_DIR)
ds_test = TccTimbreDataset(df_test, AUDIO_DIR)

dl_train = DataLoader(ds_train, batch_sampler=HardNegativeBatchSampler(ds_train, BATCH_SIZE), collate_fn=lambda b: collate_fn(b, ds_train.processor))
dl_val = DataLoader(ds_val, batch_sampler=HardNegativeBatchSampler(ds_val, BATCH_SIZE), collate_fn=lambda b: collate_fn(b, ds_val.processor))
dl_test = DataLoader(ds_test, batch_sampler=HardNegativeBatchSampler(ds_test, BATCH_SIZE), collate_fn=lambda b: collate_fn(b, ds_test.processor))

# =============================================================================
# 3. ARQUITETura DO MODELO VIRGEM
# =============================================================================
print("\nBaixando o Modelo Fundacional (LAION-CLAP) e Injetando LoRA...")
modelo_base = ClapModel.from_pretrained("laion/clap-htsat-fused").to(DEVICE)

configuracao_lora = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"], 
    lora_dropout=0.1, bias="none", modules_to_save=["logit_scale_a", "logit_scale_t"]
)
modelo_especializado = get_peft_model(modelo_base, configuracao_lora)

otimizador = AdamW(modelo_especializado.parameters(), lr=LEARNING_RATE)
monitor_parada = EarlyStopping(patience=3, min_delta=0.015, save_dir="tcc_clap_lora_timbre_elite")

# =============================================================================
# 4. LAÇO DE TREINAMENTO E VALIDAÇÃO
# =============================================================================
print("\n" + "="*80)
print("🚀 INICIANDO TREINAMENTO DEFINITIVO")
print("="*80)

for epoch in range(EPOCHS):
    modelo_especializado.train()
    loss_train_acumulada = 0
    loop = tqdm(dl_train, leave=False, desc=f"Epoch {epoch+1}/{EPOCHS} [Treino]")
    
    for batch in loop:
        otimizador.zero_grad()
        input_features = batch["input_features"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        batch_size_atual = input_features.shape[0]
        is_longer = torch.zeros((batch_size_atual, 1), dtype=torch.bool, device=DEVICE) 
        
        outputs = modelo_especializado(input_features=input_features, input_ids=input_ids, attention_mask=attention_mask, is_longer=is_longer)
        labels = torch.arange(batch_size_atual, device=DEVICE)
        loss = (F.cross_entropy(outputs.logits_per_audio, labels) + F.cross_entropy(outputs.logits_per_text, labels)) / 2.0
        
        loss.backward()
        otimizador.step()
        loss_train_acumulada += loss.item()
        loop.set_postfix(train_loss=f"{loss.item():.4f}")
        
    loss_train = loss_train_acumulada / len(dl_train)
    
    print(f"\n-> Fim da Epoch {epoch+1} | Calculando Validation Loss...")
    loss_val = avaliar_modelo(modelo_especializado, dl_val)
    print(f"   Train Loss: {loss_train:.4f}  |  Val Loss: {loss_val:.4f}")
    
    monitor_parada(loss_val, modelo_especializado, ds_train.processor)
    if monitor_parada.early_stop:
        print("\n🛑 EARLY STOPPING ACIONADO: O modelo convergiu no conjunto de Validação.")
        break

# =============================================================================
# 5. O TESTE CEGO ABSOLUTO
# =============================================================================
from peft import PeftModel
print("\n" + "="*80)
print("🎯 FASE DE TESTE CEGO: AVALIANDO O MELHOR MODELO SALVO")
print("="*80)

print("Recarregando os Pesos Ouro da pasta definitiva...")
modelo_definitivo = PeftModel.from_pretrained(modelo_base, monitor_parada.save_dir, is_trainable=False)
loss_test = avaliar_modelo(modelo_definitivo, dl_test)

print(f"\n🏆 RESULTADO FINAL PARA O TCC:")
print(f"   Loss em Dados de Teste (Nunca Vistos): {loss_test:.4f}")
print("=" * 80)

Dispositivo de processamento: cpu

Realizando Particionamento de Dados Seguro (Sem Vazamento de Pares)...
  -> Treino:    4580 amostras
  -> Validação: 570 amostras
  -> Teste:     580 amostras

Instanciando Dataloaders (Treino, Validação, Teste)...


C:\Users\victo\AppData\Local\Temp\ipykernel_3852\2570815052.py:137: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(pares_unicos)



Baixando o Modelo Fundacional (LAION-CLAP) e Injetando LoRA...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 477/477 [00:00<00:00, 6134.28it/s]



🚀 INICIANDO TREINAMENTO DEFINITIVO


...


-> Fim da Epoch 13 | Calculando Validation Loss...
   Train Loss: 1.7327  |  Val Loss: 1.7966
  ⚠️ Val Loss estagnou ou subiu. Paciência: 1/3



-> Fim da Epoch 14 | Calculando Validation Loss...
   Train Loss: 1.7422  |  Val Loss: 1.6386
  🌟 Novo Ponto de Convergência! Val Loss caiu para 1.6386.
  💾 Salvando Pesos Ouro no HD...



-> Fim da Epoch 15 | Calculando Validation Loss...
   Train Loss: 1.7179  |  Val Loss: 1.6438
  ⚠️ Val Loss estagnou ou subiu. Paciência: 1/3

🎯 FASE DE TESTE CEGO: AVALIANDO O MELHOR MODELO SALVO
Recarregando os Pesos Ouro da pasta definitiva...


C:\Users\victo\Documents\Dataset-TCC\venv_tcc\Lib\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



🏆 RESULTADO FINAL PARA O TCC:
   Loss em Dados de Teste (Nunca Vistos): 1.7626


In [12]:
# =============================================================================
# AVALIAÇÃO DE ESTADO DA ARTE: METRICS CONTRASTIVAS (RECALL@K, MRR, HARD NATIVE)
# =============================================================================
import torch
import numpy as np
import torch.nn.functional as F
from collections import defaultdict
from tqdm import tqdm

print("=====================================================================")
print("🔬 EXTRAINDO EMBEDDINGS DO CONJUNTO DE TESTE...")
print("=====================================================================")

modelo_definitivo.eval()

todos_audio_embeds = []
todos_text_embeds = []

# 1. Extração no-grad (Passo Único com extração forçada de tensores)
with torch.no_grad():
    for batch in tqdm(dl_test, desc="Computando Projeções Latentes"):
        input_features = batch["input_features"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        is_longer = torch.zeros((input_features.shape[0], 1), dtype=torch.bool, device=DEVICE)
        
        # Extraímos os objetos de saída do Hugging Face através da variação LoRA
        audio_outputs = modelo_definitivo.get_audio_features(input_features=input_features, is_longer=is_longer)
        text_outputs = modelo_definitivo.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        
        # --- EXTRAÇÃO EXTRA FORÇADA (Dicionário, Atributo ou Chave) ---
        if isinstance(audio_outputs, dict) or hasattr(audio_outputs, "keys"):
            audio_embeds = audio_outputs["audio_embeds"]
        elif hasattr(audio_outputs, "audio_embeds"):
            audio_embeds = audio_outputs.audio_embeds
        else:
            # Fallback para o primeiro elemento (pooling layer output)
            audio_embeds = audio_outputs[0]
            
        if isinstance(text_outputs, dict) or hasattr(text_outputs, "keys"):
            text_embeds = text_outputs["text_embeds"]
        elif hasattr(text_outputs, "text_embeds"):
            text_embeds = text_outputs.text_embeds
        else:
            text_embeds = text_outputs[0]
        
        # Garante que desceu para o Tensor bruto se ainda estiver envelopado
        if hasattr(audio_embeds, "audio_embeds"): audio_embeds = audio_embeds.audio_embeds
        if hasattr(text_embeds, "text_embeds"): text_embeds = text_embeds.text_embeds

        # L2 Normalization (Crucial para Cosine Similarity)
        audio_embeds = F.normalize(audio_embeds, p=2, dim=-1)
        text_embeds = F.normalize(text_embeds, p=2, dim=-1)
        
        todos_audio_embeds.append(audio_embeds.cpu())
        todos_text_embeds.append(text_embeds.cpu())

# Concatena tudo em tensores gigantes (N_amostras x 512 dimensões)
matriz_audio = torch.cat(todos_audio_embeds, dim=0)
matriz_text = torch.cat(todos_text_embeds, dim=0)
N_amostras = matriz_audio.shape[0]

print(f"✓ Embeddings extraídos com sucesso! Matriz Final: {matriz_audio.shape}\n")

# =============================================================================
# 1. MATRIZ DE SIMILARIDADE GLOBAL E RECALL@K
# =============================================================================
sim_matrix = matriz_audio @ matriz_text.T  # Shape: (N, N)

def compute_retrieval_metrics(sim_matrix):
    metrics = {}
    N = sim_matrix.shape[0]
    
    # Text-to-Audio (T2A)
    rankings_t2a = torch.argsort(sim_matrix, dim=0, descending=True)
    posicao_correta_t2a = (rankings_t2a == torch.arange(N).view(1, -1)).nonzero(as_tuple=True)[0]
    
    # Audio-to-Text (A2T)
    rankings_a2t = torch.argsort(sim_matrix, dim=1, descending=True)
    posicao_correta_a2t = (rankings_a2t == torch.arange(N).view(-1, 1)).nonzero(as_tuple=True)[1]
    
    for prefix, pos_correta in zip(['T2A', 'A2T'], [posicao_correta_t2a, posicao_correta_a2t]):
        metrics[f'{prefix}_R@1'] = (pos_correta < 1).float().mean().item() * 100
        metrics[f'{prefix}_R@5'] = (pos_correta < 5).float().mean().item() * 100
        metrics[f'{prefix}_R@10'] = (pos_correta < 10).float().mean().item() * 100
        metrics[f'{prefix}_MRR'] = (1.0 / (pos_correta.float() + 1)).mean().item()
        
    return metrics

resultados_globais = compute_retrieval_metrics(sim_matrix)

print("-" * 60)
print("🏆 MÉTRICAS GLOBAIS DE RECUPERAÇÃO (RETRIEVAL)")
print("-" * 60)
print(f"Text-to-Audio (T2A) -> O texto acha o áudio certo?")
print(f"  R@1:  {resultados_globais['T2A_R@1']:.2f}%")
print(f"  R@5:  {resultados_globais['T2A_R@5']:.2f}%")
print(f"  R@10: {resultados_globais['T2A_R@10']:.2f}%")
print(f"  MRR:  {resultados_globais['T2A_MRR']:.4f}")

print(f"\nAudio-to-Text (A2T) -> O áudio acha o texto certo?")
print(f"  R@1:  {resultados_globais['A2T_R@1']:.2f}%")
print(f"  R@5:  {resultados_globais['A2T_R@5']:.2f}%")
print(f"  R@10: {resultados_globais['A2T_R@10']:.2f}%")
print(f"  MRR:  {resultados_globais['A2T_MRR']:.4f}")


# =============================================================================
# 2. ACURÁCIA DE PARES OPOSTOS (A PROVA DO HARD NEGATIVE)
# =============================================================================
print("\n" + "-" * 60)
print("⚔️ ACURÁCIA DE CONTRASTE (HARD NEGATIVE RESOLUTION)")
print("-" * 60)

acertos = 0
total_pares_analisados = 0
df_teste_analise = ds_test.df.reset_index(drop=True)

pair_to_local_indices = defaultdict(list)
for idx, row in df_teste_analise.iterrows():
    pair_to_local_indices[row['hard_negative_pair_id']].append(idx)

for pair_id, indices in pair_to_local_indices.items():
    if len(indices) == 2:
        idx_A, idx_B = indices[0], indices[1]
        
        audio_A, text_A = matriz_audio[idx_A], matriz_text[idx_A]
        audio_B, text_B = matriz_audio[idx_B], matriz_text[idx_B]
        
        sim_A_correta = (audio_A @ text_A).item()
        sim_A_errada = (audio_A @ text_B).item()
        if sim_A_correta > sim_A_errada: acertos += 1
        
        sim_B_correta = (audio_B @ text_B).item()
        sim_B_errada = (audio_B @ text_A).item()
        if sim_B_correta > sim_B_errada: acertos += 1
        
        total_pares_analisados += 2

acuracia_hn = (acertos / total_pares_analisados) * 100 if total_pares_analisados > 0 else 0

print(f"Total de combates diretos (Áudio vs Texto Antônimo): {total_pares_analisados}")
print(f"Vitórias do Modelo (Conseguiu separar a ambiguidade): {acertos}")
print(f"\n🔥 ACURÁCIA HARD NEGATIVE: {acuracia_hn:.2f}%")
print("=====================================================================")

🔬 EXTRAINDO EMBEDDINGS DO CONJUNTO DE TESTE...


Computando Projeções Latentes:   0%|                                                            | 0/19 [00:04<?, ?it/s]


KeyError: 'audio_embeds'